In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

import pandas as pd
from src.utils import setup_logging
from src.risk_metrics import RiskEngine

In [2]:
# 1. Initialize the logging engine
logger = setup_logging()
logger.info("Starting Phase 4: Core Risk Metrics Analysis.")

2026-07-12 23:16:59 [INFO] utils: Logging infrastructure successfully initialized.
2026-07-12 23:16:59 [INFO] 2956508608: Starting Phase 4: Core Risk Metrics Analysis.


Writing logs to: /Users/admin/Desktop/Portofolio Analyzer/logs/portfolio_analyzer.log


In [3]:
# 2. Ingest processed asset matrices from disk
cleaned_prices = pd.read_csv("data/processed/cleaned_prices.csv", index_col="Date", parse_dates=True)
simple_returns = pd.read_csv("data/processed/simple_returns.csv", index_col="Date", parse_dates=True)

In [4]:
# 3. Instantiate the quantitative Risk Engine
risk_engine = RiskEngine()


In [5]:
# 4. Loop through assets and calculate tail risks
risk_summary = {}

for asset in simple_returns.columns:
    # Gather historical prices and return vectors for the specific asset
    asset_prices = cleaned_prices[asset]
    asset_returns = simple_returns[asset]
    
    # Run risk engine functions
    max_dd = risk_engine.calculate_max_drawdown(asset_prices)
    var_95, cvar_95 = risk_engine.calculate_historical_var_cvar(asset_returns, confidence_level=0.95)
    
    # Store results (Formatting as percentages for presentation mapping)
    risk_summary[asset] = {
        "Maximum Drawdown": f"{max_dd * 100:.2f}%",
        "Daily VaR (95%)": f"{var_95 * 100:.2f}%",
        "Daily Expected Shortfall (95%)": f"{cvar_95 * 100:.2f}%"
    }

2026-07-12 23:16:59 [INFO] risk_metrics: Calculating Maximum Drawdown for series: AAPL
2026-07-12 23:16:59 [INFO] risk_metrics: Calculating Historical VaR/CVaR (95.0%) for: AAPL
2026-07-12 23:16:59 [INFO] risk_metrics: Calculating Maximum Drawdown for series: GLD
2026-07-12 23:16:59 [INFO] risk_metrics: Calculating Historical VaR/CVaR (95.0%) for: GLD
2026-07-12 23:16:59 [INFO] risk_metrics: Calculating Maximum Drawdown for series: IEF
2026-07-12 23:16:59 [INFO] risk_metrics: Calculating Historical VaR/CVaR (95.0%) for: IEF
2026-07-12 23:16:59 [INFO] risk_metrics: Calculating Maximum Drawdown for series: MSFT
2026-07-12 23:16:59 [INFO] risk_metrics: Calculating Historical VaR/CVaR (95.0%) for: MSFT


In [6]:
# 5. Convert to DataFrame and display
risk_df = pd.DataFrame(risk_summary).T
print("\n=== Asset Risk Profiles ===")
display(risk_df)


=== Asset Risk Profiles ===


,Maximum Drawdown,Daily VaR (95%),Daily Expected Shortfall (95%)
AAPL,-38.52%,2.76%,4.18%
GLD,-22.00%,1.48%,2.11%
IEF,-23.92%,0.65%,0.92%
MSFT,-37.15%,2.63%,3.81%


In [7]:
from src.visualization import FinancialVisualizer

# Initialize visualizer engine
visualizer = FinancialVisualizer()

# Generate and save down the comparative drawdown curves
visualizer.plot_portfolio_drawdowns(cleaned_prices)
print("Risk plots generated successfully and stored in the images/ directory.")

2026-07-12 23:17:00 [INFO] visualization: Generating historical drawdown curve charts.
2026-07-12 23:17:00 [INFO] visualization: Drawdown charts successfully exported to: /Users/admin/Desktop/Portofolio Analyzer/images/asset_drawdowns.png


Risk plots generated successfully and stored in the images/ directory.


In [8]:
risk_summary = {}
# Assuming an annualized Risk-Free rate environment of 4.0% (0.04)
risk_free_assumption = 0.04 

for asset in simple_returns.columns:
    asset_prices = cleaned_prices[asset]
    asset_returns = simple_returns[asset]
    
    # Run the expanded Risk Engine portfolio
    max_dd = risk_engine.calculate_max_drawdown(asset_prices)
    var_95, cvar_95 = risk_engine.calculate_historical_var_cvar(asset_returns, confidence_level=0.95)
    sharpe = risk_engine.calculate_sharpe_ratio(asset_returns, risk_free_assumption)
    sortino = risk_engine.calculate_sortino_ratio(asset_returns, risk_free_assumption)
    
    risk_summary[asset] = {
        "Annualized Sharpe Ratio": f"{sharpe:.2f}",
        "Annualized Sortino Ratio": f"{sortino:.2f}",
        "Maximum Drawdown": f"{max_dd * 100:.2f}%",
        "Daily VaR (95%)": f"{var_95 * 100:.2f}%",
        "Daily Expected Shortfall (95%)": f"{cvar_95 * 100:.2f}%"
    }

# Render updated summary frame
risk_df = pd.DataFrame(risk_summary).T
print("\n=== Expanded Asset Risk-Return Profiles ===")
display(risk_df)

2026-07-12 23:17:00 [INFO] risk_metrics: Calculating Maximum Drawdown for series: AAPL
2026-07-12 23:17:00 [INFO] risk_metrics: Calculating Historical VaR/CVaR (95.0%) for: AAPL
2026-07-12 23:17:00 [INFO] risk_metrics: Calculating Annualized Sharpe Ratio for: AAPL
2026-07-12 23:17:00 [INFO] risk_metrics: Calculating Annualized Sortino Ratio for: AAPL
2026-07-12 23:17:00 [INFO] risk_metrics: Calculating Maximum Drawdown for series: GLD
2026-07-12 23:17:00 [INFO] risk_metrics: Calculating Historical VaR/CVaR (95.0%) for: GLD
2026-07-12 23:17:00 [INFO] risk_metrics: Calculating Annualized Sharpe Ratio for: GLD
2026-07-12 23:17:00 [INFO] risk_metrics: Calculating Annualized Sortino Ratio for: GLD
2026-07-12 23:17:00 [INFO] risk_metrics: Calculating Maximum Drawdown for series: IEF
2026-07-12 23:17:00 [INFO] risk_metrics: Calculating Historical VaR/CVaR (95.0%) for: IEF
2026-07-12 23:17:00 [INFO] risk_metrics: Calculating Annualized Sharpe Ratio for: IEF
2026-07-12 23:17:00 [INFO] risk_metr


=== Expanded Asset Risk-Return Profiles ===


,Annualized Sharpe Ratio,Annualized Sortino Ratio,Maximum Drawdown,Daily VaR (95%),Daily Expected Shortfall (95%)
AAPL,0.85,1.25,-38.52%,2.76%,4.18%
GLD,0.72,1.03,-22.00%,1.48%,2.11%
IEF,-0.39,-0.54,-23.92%,0.65%,0.92%
MSFT,0.85,1.25,-37.15%,2.63%,3.81%
